In [ ]:
#import os
#os.environ["LANGCHAIN_TRACING_V2"] = "true"
#os.environ["LANGCHAIN_API_KEY"] = "你的langchain_key"

#os.environ["OPENAI_API_KEY"] = "你的openapi key"
#os.environ["OPENAI_BASE_URL"] = "你的openai 请求地址"

In [ ]:
#pip install  langchain pypdf

In [1]:
# document loader
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("docs_for_9_RAG_basic/DeepSeek_R1.pdf")

pages = []
# 异步加载前 11 页
i = 0  # 手动维护计数器
async for page in loader.alazy_load():
    if i >= 11:  # 只加载前 11 页
        break
    pages.append(page)
    i+=1
    
print(f"{pages[0].metadata}\n")
print(pages[0].page_content)

{'producer': 'macOS 版本13.4（版号22F66） Quartz PDFContext, AppendMode 1.1', 'creator': 'LaTeX with hyperref', 'creationdate': "D:20250123075355Z00'00'", 'author': '', 'subject': '', 'trapped': '/False', 'title': '', 'moddate': "D:20250130140144Z00'00'", 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'keywords': '', 'source': 'docs_for_9_RAG_basic/DeepSeek_R1.pdf', 'total_pages': 22, 'page': 0, 'page_label': '1'}

DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via
Reinforcement Learning
DeepSeek-AI
research@deepseek.com
Abstract
We introduce our first-generation reasoning models, DeepSeek-R1-Zero and DeepSeek-R1.
DeepSeek-R1-Zero, a model trained via large-scale reinforcement learning (RL) without super-
vised fine-tuning (SFT) as a preliminary step, demonstrates remarkable reasoning capabilities.
Through RL, DeepSeek-R1-Zero naturally emerges with numerous powerful and intriguing
reasoning behaviors. However, it encount

In [ ]:
#pip install tiktoken

In [2]:
# text spliter
# 模型类型
# 建议使用官方推荐的第二代嵌入模型：text-embedding-ada-002
# cl100k_base 对应的分词器（TOKENIZER）
from langchain_text_splitters import CharacterTextSplitter
text_splitter_0 = CharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base", chunk_size=20, chunk_overlap=0
)
texts_0 = text_splitter_0.split_text(pages[0].page_content)
print(texts_0[0])

DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via
Reinforcement Learning
DeepSeek-AI
research@deepseek.com
Abstract
We introduce our first-generation reasoning models, DeepSeek-R1-Zero and DeepSeek-R1.
DeepSeek-R1-Zero, a model trained via large-scale reinforcement learning (RL) without super-
vised fine-tuning (SFT) as a preliminary step, demonstrates remarkable reasoning capabilities.
Through RL, DeepSeek-R1-Zero naturally emerges with numerous powerful and intriguing
reasoning behaviors. However, it encounters challenges such as poor readability, and language
mixing. To address these issues and further enhance reasoning performance, we introduce
DeepSeek-R1, which incorporates multi-stage training and cold-start data before RL. DeepSeek-
R1 achieves performance comparable to OpenAI-o1-1217 on reasoning tasks. To support the
research community, we open-source DeepSeek-R1-Zero, DeepSeek-R1, and six dense models
(1.5B, 7B, 8B, 14B, 32B, 70B) distilled from DeepSeek-R1 based o

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # 每个片段的最大字符数
    chunk_overlap=100,  # 片段之间的重叠字符数
    separators=["\n\n", "\n"]  # 分割符（按段落、句子、标点等）
)
chunks = text_splitter.split_documents(pages)
print(f"文档被分割成 {len(chunks)} 个块")

文档被分割成 37 个块


In [4]:
print(chunks[0])

page_content='DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via
Reinforcement Learning
DeepSeek-AI
research@deepseek.com
Abstract
We introduce our first-generation reasoning models, DeepSeek-R1-Zero and DeepSeek-R1.
DeepSeek-R1-Zero, a model trained via large-scale reinforcement learning (RL) without super-
vised fine-tuning (SFT) as a preliminary step, demonstrates remarkable reasoning capabilities.
Through RL, DeepSeek-R1-Zero naturally emerges with numerous powerful and intriguing
reasoning behaviors. However, it encounters challenges such as poor readability, and language
mixing. To address these issues and further enhance reasoning performance, we introduce
DeepSeek-R1, which incorporates multi-stage training and cold-start data before RL. DeepSeek-
R1 achieves performance comparable to OpenAI-o1-1217 on reasoning tasks. To support the
research community, we open-source DeepSeek-R1-Zero, DeepSeek-R1, and six dense models' metadata={'producer': 'macOS 版本13.4（版号22F66） Quar

In [ ]:
#pip install langchain_openai

In [5]:
# from langchain_openai import OpenAIEmbeddings
from langchain.embeddings import HuggingFaceEmbeddings

# 初始化 OpenAI 嵌入模型
# embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")

# embeddings = HuggingFaceEmbeddings()
embeddings = HuggingFaceEmbeddings(
    model_name="/home/haoyu/work/model/bge-m3",
    model_kwargs={"device": "cuda"},
)

# 取第一个片段进行embedding，省token
document_embeddings = embeddings.embed_documents(chunks[0].page_content)
print("文档嵌入：", document_embeddings[0])

/tmp/ipykernel_896730/3124378524.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/home/haoyu/anaconda3/envs/langchain/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


文档嵌入： [0.0007608570158481598, 0.055348243564367294, -0.05628827586770058, -0.009958340786397457, -0.025282632559537888, -3.2163807190954685e-06, -0.0006404502200894058, 0.030161716043949127, 0.020964233204722404, -0.0018598895985633135, 0.0018848201725631952, 0.006762093398720026, 0.024208473041653633, -0.02753618359565735, 0.002977101830765605, -0.059899356216192245, -0.03261630982160568, 0.002610756317153573, 0.014550969935953617, -0.03471504896879196, -0.016425954177975655, -0.017865069210529327, 0.0052064391784369946, 0.006491484120488167, -0.03302876651287079, -0.016482427716255188, -0.0016310774954035878, 0.014748918823897839, -0.028713911771774292, 0.039987560361623764, -0.0025754498783499002, -0.03985271975398064, -0.038272347301244736, -0.08184656500816345, 0.00037177526974119246, -0.04130556806921959, -0.032393936067819595, -0.011172175407409668, -0.12102305144071579, -0.002423750003799796, 0.02966107241809368, -0.011824209243059158, 0.011618311516940594, 0.013570363633334637

In [6]:
print(len(document_embeddings))

935


In [7]:
# 单个查询
query = "这篇文章介绍DeepSeek的什么版本"
# 使用 embed_query 嵌入单个查询
query_embedding = embeddings.embed_query(query)
print("查询嵌入：", query_embedding)

查询嵌入： [-0.03304440528154373, -0.02198326215147972, -0.012846897356212139, 0.009360109455883503, 0.002911279210820794, -0.013064149767160416, 0.07195574790239334, -0.019393673166632652, 0.024478040635585785, 0.014070263132452965, 0.0014559265691787004, -0.0023326713126152754, -0.030432168394327164, 0.003198757767677307, -0.0009109922684729099, -0.015143252909183502, -0.043891068547964096, -0.021643320098519325, 0.0107648316770792, -0.01353350467979908, -0.024804092943668365, -0.008729442954063416, 0.026309827342629433, 0.05230272561311722, -0.0008464538841508329, 0.054890718311071396, 0.03363867849111557, -0.028848795220255852, 0.012875165790319443, -0.023908114060759544, 0.01397386658936739, -0.018832840025424957, 0.011648617684841156, -0.056854814291000366, -0.045071545988321304, -0.042125072330236435, 0.016663769260048866, -0.014005602337419987, -0.037227991968393326, 0.014984712935984135, 0.03795269504189491, -0.010924856178462505, 0.035862699151039124, -0.017923861742019653, 0.0150

In [8]:
# vector store
from langchain_core.vectorstores import InMemoryVectorStore
# 实例化向量存储
vector_store = InMemoryVectorStore(embeddings)
# 将已经转为向量的文档存储到向量存储中
ids =vector_store.add_documents(documents=chunks)
print(ids)
#vector_store.delete(ids=["93ed319a-5110-40ac-8bf4-117a220ed0cb"])


['904cdd43-e19f-4659-b040-ef0237ce7e11', '69e35f8c-dde6-437c-8a30-6d154cccbd4d', '19c0c2c3-7197-446c-b5c2-4ec9d3a3bc96', '99468ccb-948b-4551-9f25-3df4a722b8ef', 'bf1618fe-2d2b-46df-b00f-6da8ac2104d0', 'b82784cc-5a9a-44ed-8902-8b6b2a744fb9', '742c04c1-9d55-490d-b11d-a54d67e30b90', '4c520629-89f8-40d9-8bf2-e5dffd0e22e9', '61e0360a-da3f-4b9c-b121-0d3ccc18b31b', '06c33655-a257-4953-ac77-de513a49ae00', 'd31b3093-f634-4d39-b9cb-8c636c2d785d', '11520c07-fd67-44bb-bd2e-9dc8fe5b1e37', 'cb88bc95-c37d-40ac-87d3-1e35e94647ed', '5afc54d1-13ac-4d1c-83fc-d081c01a7318', '553d320d-d2ce-4d40-a360-8be627bb68e9', 'd4ada4da-d879-4b67-9b59-b03e77113bbe', '011c7d06-2b5a-4270-9ae1-cdc244fc7c0d', '2388566d-e497-406d-9d51-81c5dd59a395', '0f73c179-0379-4f63-9c9f-7e8c02df3153', 'bb7683b8-4d47-40db-bf31-fcc296e3239b', '6edf29b3-a233-40ac-98e9-8eaf9552ba1d', 'd2b81ae1-f079-41e1-a028-5891650091da', '6f06737d-994e-4119-8b99-db3f27042749', '1fd4d04a-c0c3-407c-8715-87b59e17ca95', '70c4b922-e9a9-489d-be33-996f8365059f',

In [9]:
vector_store.similarity_search('What is the model introduced in this paper?',k=4)

[Document(id='19c0c2c3-7197-446c-b5c2-4ec9d3a3bc96', metadata={'producer': 'macOS 版本13.4（版号22F66） Quartz PDFContext, AppendMode 1.1', 'creator': 'LaTeX with hyperref', 'creationdate': "D:20250123075355Z00'00'", 'author': '', 'subject': '', 'trapped': '/False', 'title': '', 'moddate': "D:20250130140144Z00'00'", 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'keywords': '', 'source': 'docs_for_9_RAG_basic/DeepSeek_R1.pdf', 'total_pages': 22, 'page': 1, 'page_label': '2'}, page_content='Contents\n1 Introduction 3\n1.1 Contributions . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 4\n1.2 Summary of Evaluation Results . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 4\n2 Approach 5\n2.1 Overview . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 5\n2.2 DeepSeek-R1-Zero: Reinforcement Learning on the Base Model . . . . . . . . . . 5\n2.2.1 Reinforcement Learnin

In [10]:
# 将向量存储转换为检索器
retriever = vector_store.as_retriever()
retriever.invoke('What is the model introduced in this paper?')

[Document(id='19c0c2c3-7197-446c-b5c2-4ec9d3a3bc96', metadata={'producer': 'macOS 版本13.4（版号22F66） Quartz PDFContext, AppendMode 1.1', 'creator': 'LaTeX with hyperref', 'creationdate': "D:20250123075355Z00'00'", 'author': '', 'subject': '', 'trapped': '/False', 'title': '', 'moddate': "D:20250130140144Z00'00'", 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'keywords': '', 'source': 'docs_for_9_RAG_basic/DeepSeek_R1.pdf', 'total_pages': 22, 'page': 1, 'page_label': '2'}, page_content='Contents\n1 Introduction 3\n1.1 Contributions . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 4\n1.2 Summary of Evaluation Results . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 4\n2 Approach 5\n2.1 Overview . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 5\n2.2 DeepSeek-R1-Zero: Reinforcement Learning on the Base Model . . . . . . . . . . 5\n2.2.1 Reinforcement Learnin